# Interactive Training and Learning

This notebook demonstrates the acetate learning workflow interactively.
It can retrain the LGP surrogates, rerun posterior sampling, export
validation-ready parameter samples, and generate a couple of posterior
plots.

In [ ]:
from pathlib import Path

from bff.bayes.results import PosteriorResults
from bff.io.logs import Logger
from bff.plotting import plot_corner, plot_marginals
from bff.workflows.fit import main as fit_workflow
from bff.workflows.learn.main import build_problem

ROOT = Path('.').resolve()
FIT_CONFIG = ROOT.parent / 'configs' / 'fit.yaml'
SPECS = ROOT.parent / '03-sample' / 'specs.yaml'
MODEL_PATHS = {
    'rdf': ROOT.parent / '05-fit' / 'models' / 'rdf.lgp',
    'hb': ROOT.parent / '05-fit' / 'models' / 'hb.lgp',
    'dist': ROOT.parent / '05-fit' / 'models' / 'dist.lgp',
}
MCMC_OPTIONS = {
    'priors_disttype': 'normal',
    'total_steps': 10000,
    'warmup': 2000,
    'progress_stride': 1000,
    'fn_checkpoint': ROOT / 'mcmc-checkpoint.pt',
    'fn_posterior': ROOT / 'posterior.pt',
    'fn_priors': ROOT / 'priors.pt',
    'restart': False,
    'device': 'cuda',
    'rhat_tol': 1.05,
}

ROOT, FIT_CONFIG, SPECS, MODEL_PATHS

## 1. Retrain the surrogate models

Re-run this cell if you changed any dataset or training settings.

In [ ]:
fit_workflow(FIT_CONFIG)

## 2. Build and run posterior inference

This mirrors `bff.workflows.learn.main()` directly: the notebook
builds the `InferenceProblem` from `specs.yaml` and the selected
`.lgp` files without loading a learn config file.

In [ ]:
logger = Logger('BFF-notebook', mode='w')
problem = build_problem(
    specs=SPECS,
    model_paths=MODEL_PATHS,
)
results = problem.infer(
    logger=logger,
    **MCMC_OPTIONS,
)

## 3. Load, prepare, and export posterior samples

The exported `posterior-samples.yaml` can be used directly by the
`07-validate` stage.

In [ ]:
results = PosteriorResults.load(
    posterior=ROOT / 'posterior.pt',
    priors=ROOT / 'priors.pt',
    specs=SPECS,
)
results.prepare_samples()
results.sample_parameters(
    n_samples=10,
    fn_out=ROOT / 'posterior-samples.yaml',
    overwrite=True,
)
results.map_estimates

## 4. Plot the posterior

The figures are saved next to the notebook and displayed inline.

In [ ]:
plot_corner(results)
plot_marginals(results, SPECS)